# PROJECT: Fine-tune ResNet-50 on Custom Data

Reach for this when you need: 
- Complete Transfer Learning boilerplate for Computer Vision.
- Reference for discriminative learning rates and unfreezing strategies.
- Implementing standard image classification with a pre-trained backbone.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import models, transforms, datasets
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 1. Data Pipeline
Using ImageNet normalization as the model was pretrained on ImageNet.

In [ ]:
tform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Using CIFAR-10 as a proxy for 'custom data'
train_ds = datasets.CIFAR10(root='./data', train=True, download=True, transform=tform)
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)

test_ds = datasets.CIFAR10(root='./data', train=False, download=True, transform=tform)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False)

## 2. Model Setup (Transfer Learning)

| Strategy | Action | Purpose |
| :--- | :--- | :--- |
| **Freezing** | `requires_grad = False` | Faster initial training, preserves learned features |
| **Head Replace** | `model.fc = Linear(...)` | Adapting to custom class count |
| **Unfreezing** | `requires_grad = True` | Final optimization of feature extractors |

In [ ]:
model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)

# Phase 1: Freeze backbone
for param in model.parameters():
    param.requires_grad = False

# Replace head
model.fc = nn.Linear(model.fc.in_features, 10)
model = model.to(device)

optimizer = optim.AdamW(model.fc.parameters(), lr=1e-3, weight_decay=1e-3)
criterion = nn.CrossEntropyLoss()

## 3. Fine-tuning with Discriminative Learning Rates

✅ **Use when**: After the head is converged, update the backbone slowly to adapt to specific textures of new data.
❌ **Don't use when**: Data is too small (overfitting) or very similar to ImageNet.

In [ ]:
# Phase 2: Unfreeze bottom layers
for param in model.layer4.parameters():
    param.requires_grad = True

# Discriminative Learning Rates: Layer4 (lower LR), Head (higher LR)
optimizer = optim.AdamW([
    {'params': model.layer4.parameters(), 'lr': 1e-5},
    {'params': model.fc.parameters(), 'lr': 1e-4}
], weight_decay=1e-2)

scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

### Common Pitfalls
- **Normalization**: Forgetting ImageNet normalization (`mean=[0.485, ...]`) reduces accuracy even if the head is trained well.
- **LR Scale**: Setting a high LR on the backbone during unfreezing can destroy pretrained features ("Catastrophic Forgetting").
- **Batching**: Large images (224x224) consume significant VRAM; use `accumulation_steps` if OOM.

## 4. Training Loop

Two-phase training:

| Phase | Frozen? | Optimizer targets | Epochs |
| :--- | :--- | :--- | :--- |
| **Phase 1** – Head warmup | Backbone frozen | `fc` only | 3–5 |
| **Phase 2** – Fine-tune | `layer4` + `fc` unfrozen | DLR (1e-5 / 1e-4) | 5–10 |

> Re-run the model setup cell before Phase 1 and the DLR cell before Phase 2.

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for images, labels in tqdm(loader, desc='Train', leave=False):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)
        correct += (logits.argmax(1) == labels).sum().item()
        total += images.size(0)

    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for images, labels in tqdm(loader, desc='Val  ', leave=False):
        images, labels = images.to(device), labels.to(device)
        logits = model(images)
        loss = criterion(logits, labels)

        total_loss += loss.item() * images.size(0)
        correct += (logits.argmax(1) == labels).sum().item()
        total += images.size(0)

    return total_loss / total, correct / total

In [ ]:
# ── Phase 1: Train head only (re-run Model Setup cell first) ──────────────
PHASE1_EPOCHS = 5

# Reset to frozen backbone + fresh head optimizer
model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
for param in model.parameters():
    param.requires_grad = False
model.fc = nn.Linear(model.fc.in_features, 10) # default requires_grad = True
model = model.to(device)
optimizer_p1 = optim.AdamW(model.fc.parameters(), lr=1e-3, weight_decay=1e-3)
criterion = nn.CrossEntropyLoss()

print('=== Phase 1: Head-only training ===')
for epoch in range(1, PHASE1_EPOCHS + 1):
    tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer_p1, criterion, device)
    val_loss, val_acc = evaluate(model, test_loader, criterion, device)
    print(f'Epoch {epoch}/{PHASE1_EPOCHS} | '
          f'Train loss: {tr_loss:.4f}  acc: {tr_acc:.3f} | '
          f'Val loss: {val_loss:.4f}  acc: {val_acc:.3f}')

In [ ]:
# ── Phase 2: Unfreeze layer4, use Discriminative LR + Cosine scheduler ───
PHASE2_EPOCHS = 10

for param in model.layer4.parameters():
    param.requires_grad = True

optimizer_p2 = optim.AdamW([
    {'params': model.layer4.parameters(), 'lr': 1e-5},
    {'params': model.fc.parameters(),     'lr': 1e-4}
], weight_decay=1e-2)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer_p2, T_max=PHASE2_EPOCHS)

print('=== Phase 2: Fine-tuning with DLR ===')
best_val_acc = 0.0
for epoch in range(1, PHASE2_EPOCHS + 1):
    tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer_p2, criterion, device)
    val_loss, val_acc = evaluate(model, test_loader, criterion, device)
    scheduler.step()

    flag = ' ✓ best' if val_acc > best_val_acc else ''
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_resnet50.pt')  # checkpoint

    print(f'Epoch {epoch}/{PHASE2_EPOCHS} | '
          f'Train loss: {tr_loss:.4f}  acc: {tr_acc:.3f} | '
          f'Val loss: {val_loss:.4f}  acc: {val_acc:.3f}{flag}')

print(f'\nBest validation accuracy: {best_val_acc:.3f}')

## 5. CLIP-style Contrastive Loss (Optional Extension)

CLIP loss is **not** part of standard supervised fine-tuning, but it is a useful concept to understand and can be applied here as a **self-supervised regularizer** using **image augmentation pairs** instead of image-text pairs.

### How it works
Given a batch of `N` images, produce **two augmented views** of each. The model encodes both into embeddings. CLIP loss pulls the two views of the same image together and pushes all other pairs apart — a form of contrastive self-supervision.

| Pair | Target |
| :--- | :--- |
| Same image, different crop | Maximize similarity |
| Different images | Minimize similarity |

```
 Image → [aug1]──► Encoder ──► z_i ─┐
                                      ├── cosine sim matrix → symmetric CE loss
        → [aug2]──► Encoder ──► z_j ─┘
```

> **Note**: This is **SimCLR-style** contrastive loss, which is the same symmetric cross-entropy objective CLIP uses. True CLIP trains with image-text pairs; here we use augmentation pairs as a vision-only analogue.

In [ ]:
# ── CLIP-style contrastive loss implementation ────────────────────────────

class ContrastiveProjection(nn.Module):
    """MLP head that maps ResNet features to a contrastive embedding space."""
    def __init__(self, in_dim: int, proj_dim: int = 128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, in_dim),
            nn.ReLU(),
            nn.Linear(in_dim, proj_dim)
        )

    def forward(self, x):
        return F.normalize(self.net(x), dim=-1)  # L2-normalized


def clip_loss(z_i: torch.Tensor, z_j: torch.Tensor, temperature: float = 0.07):
    """
    Symmetric cross-entropy contrastive loss (CLIP / SimCLR objective).

    Args:
        z_i, z_j:    L2-normalized embeddings, shape (N, D)
        temperature: scaling factor; lower → sharper distribution

    Returns:
        Scalar loss
    """
    N = z_i.size(0)
    # Cosine similarity matrix: (N, N)
    logits = torch.matmul(z_i, z_j.T) / temperature   # (N, N)

    # Diagonal entries are positive pairs (same image, different augmentation)
    labels = torch.arange(N, device=z_i.device)       # [0, 1, 2, ..., N-1]

    # Symmetric loss: rows predict columns AND columns predict rows
    loss_i = F.cross_entropy(logits,   labels)         # image-to-image
    loss_j = F.cross_entropy(logits.T, labels)         # augmentation-to-image

    return (loss_i + loss_j) / 2

In [ ]:
# ── Augmentation pair pipeline for contrastive training ───────────────────

contrastive_tform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.2, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.4, 0.4, 0.4, 0.1),
    transforms.RandomGrayscale(p=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

class TwoCropDataset(torch.utils.data.Dataset):
    """Returns two independently augmented views of each image."""
    def __init__(self, dataset, transform):
        self.dataset = dataset
        self.transform = transform

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        img, label = self.dataset[idx]
        # dataset returns PIL images if transform=None was passed at construction
        # here we re-apply our own transform twice for two random views
        view1 = self.transform(img)
        view2 = self.transform(img)
        return view1, view2, label


# Use raw CIFAR-10 (no transform) so TwoCropDataset applies its own
raw_train_ds = datasets.CIFAR10(root='./data', train=True, download=True, transform=None)
contrastive_ds = TwoCropDataset(raw_train_ds, contrastive_tform)
contrastive_loader = DataLoader(contrastive_ds, batch_size=256, shuffle=True, drop_last=True)

In [ ]:
# ── Contrastive pre-training loop (vision-only CLIP objective) ────────────
#
# Architecture: ResNet-50 backbone (no fc) → ContrastiveProjection head
# After pre-training, discard projection head and attach classification fc
# for supervised fine-tuning (standard transfer learning workflow).

CONTRASTIVE_EPOCHS = 5
TEMPERATURE = 0.07

# Build encoder: ResNet-50 minus the classification head
encoder = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
feature_dim = encoder.fc.in_features          # 2048
encoder.fc = nn.Identity()                    # remove classification head
encoder = encoder.to(device)

projector = ContrastiveProjection(in_dim=feature_dim, proj_dim=128).to(device)

contrastive_opt = optim.AdamW(
    list(encoder.parameters()) + list(projector.parameters()),
    lr=1e-4, weight_decay=1e-4
)

print('=== Contrastive Pre-training (CLIP-style loss) ===')
for epoch in range(1, CONTRASTIVE_EPOCHS + 1):
    encoder.train(); projector.train()
    epoch_loss = 0.0

    for view1, view2, _ in tqdm(contrastive_loader, desc=f'Epoch {epoch}', leave=False):
        view1, view2 = view1.to(device), view2.to(device)

        z_i = projector(encoder(view1))   # (N, 128)
        z_j = projector(encoder(view2))   # (N, 128)

        loss = clip_loss(z_i, z_j, temperature=TEMPERATURE)

        contrastive_opt.zero_grad()
        loss.backward()
        contrastive_opt.step()

        epoch_loss += loss.item()

    print(f'Epoch {epoch}/{CONTRASTIVE_EPOCHS} | Contrastive Loss: {epoch_loss / len(contrastive_loader):.4f}')


# ── Transfer: attach classification head after contrastive pre-training ───
encoder.fc = nn.Linear(feature_dim, 10)
encoder = encoder.to(device)
# Now feed `encoder` into the Phase 1 / Phase 2 training loop above

### Common Pitfalls
- **Normalization**: Forgetting ImageNet normalization (`mean=[0.485, ...]`) reduces accuracy even if the head is trained well.
- **LR Scale**: Setting a high LR on the backbone during unfreezing can destroy pretrained features ("Catastrophic Forgetting").
- **Batching**: Large images (224x224) consume significant VRAM; use `accumulation_steps` if OOM.

### Key Takeaways
- Standard fine-tuning involves training a new head first, then unfreezing layers gradually.
- Discriminative Learning Rates ensure the head adapts quickly while the backbone updates gently.
- Always log validation accuracy to detect the point where fine-tuning starts to overfit.
- CLIP-style contrastive loss (symmetric CE on cosine similarity) can serve as a self-supervised pre-training objective using augmentation pairs when labels are scarce.